# 02 — Visualize Annotations

**Purpose:** Draw bounding boxes on document images and visually confirm annotations are correct.

**Why mandatory:** If boxes are wrong here, your detector will learn wrong things.
Never train blindly without looking at your annotations first.

Color scheme:
- 🟢 Green = handwritten
- 🔴 Red = printed
- 🟠 Orange = formula
- 🟣 Magenta = table
- 🔵 Cyan = annotation
- ⬜ Gray = image
- 🟣 Purple = graph

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ── Load records from local JSONL (faster than HF) ───────────────
# Requires: python -m src.data.load_dataset --splits train --save-jsonl

from src.utils.jsonl import read_jsonl
from src.utils.paths import get_path

jsonl_path = get_path('processed_root') / 'train_raw.jsonl'

if jsonl_path.exists():
    records = read_jsonl(jsonl_path)
    print(f'Loaded {len(records):,} records from local JSONL.')
else:
    print('Local JSONL not found. Loading from Hugging Face (slower)...')
    from src.data.load_dataset import load_split
    ds = load_split('train')
    records = list(ds)
    print(f'Loaded {len(records):,} records from HF.')

In [ ]:
# ── Save 10 annotated debug images to outputs/debug_images/ ───────
# This is the MANDATORY visual inspection step.

from src.visualization.draw_bboxes import run

run(records, n_samples=10)

In [ ]:
# ── Display saved debug images inline ────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

debug_dir = get_path('debug_images_dir')
debug_imgs = sorted(debug_dir.glob('*_annotated.png'))[:6]

if not debug_imgs:
    print('No debug images found. Run the cell above first.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    for ax, img_path in zip(axes, debug_imgs):
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(img_path.name[:40], fontsize=8)
        ax.axis('off')
    for ax in axes[len(debug_imgs):]:
        ax.axis('off')
    plt.suptitle('Annotation Debug Images — Check boxes align with document content', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f'\nShowing {len(debug_imgs)} of {len(sorted(debug_dir.glob("*_annotated.png")))} saved images.')

In [ ]:
# ── Draw a single sample interactively ───────────────────────────
# Change sample_idx to inspect a specific document

import json
import numpy as np
from PIL import Image
from src.visualization.draw_bboxes import draw_bboxes_on_image

sample_idx = 2   # ← change this to explore different documents
rec = records[sample_idx]

img_path = get_path('raw_train_images') / rec['file_name']
if img_path.exists():
    pil_img = Image.open(img_path)
else:
    # Blank canvas with bbox overlay (when images not downloaded)
    w = rec.get('image_width', 800)
    h = rec.get('image_height', 1000)
    pil_img = Image.new('RGB', (w, h), (245, 245, 245))

regions = rec.get('regions', [])
if isinstance(regions, str):
    regions = json.loads(regions)

drawn = draw_bboxes_on_image(pil_img, regions)

plt.figure(figsize=(14, 10))
plt.imshow(drawn[:, :, ::-1])   # BGR → RGB for matplotlib
plt.title(f"Sample {sample_idx}: {rec.get('file_name')}  source={rec.get('source')}  regions={len(regions)}",
          fontsize=10)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── What to look for ──────────────────────────────────────────────
print('''
VISUAL INSPECTION CHECKLIST:

  ✅ Boxes tightly surround the actual text/region
  ✅ Colors match the correct region type
  ✅ No boxes floating in empty white space
  ✅ Handwriting regions are green
  ✅ Printed text regions are red
  ✅ Tables have magenta borders around the full table

  ⚠️  Watch for:
     - Boxes that cover multiple regions (should be split)
     - Boxes labeled wrong type (e.g. printed labeled handwritten)
     - Boxes with [!] marker = illegible — these are excluded from crops
''')

## ✅ Next Step

```bash
# Generate dataset statistics:
python -m src.data.dataset_stats --from-jsonl data/processed/train_raw.jsonl

# Then open:
# notebooks/03_prepare_crops.ipynb
```